# Chapitre 9 - Lab : validation à deux phases avec PostgreSQL

PostgreSQL expose les deux phases de 2PC aux applications : `PREPARE TRANSACTION 'id'`
(phase 1 : le participant écrit tout sur disque et promet de pouvoir valider) puis
`COMMIT PREPARED 'id'` ou `ROLLBACK PREPARED 'id'` (phase 2). C'est la brique qu'utilisent
les gestionnaires de transactions XA. Ce lab écrit un **coordinateur** minimal en Python
entre deux bases (deux banques), puis observe sur le vrai système ce que le cours a
simulé : transactions en doute, verrous tenus, survie à un redémarrage, reprise par le
journal du coordinateur, décision heuristique, et coût en latence.

**Démarrer et arrêter PostgreSQL** (depuis la racine du dépôt ; `max_prepared_transactions`
est activé dans le `compose.yml`) :

```bash
docker compose -f docker/transactions/compose.yml -p ddia_transactions up -d
# ... exécuter le notebook ...
docker compose -f docker/transactions/compose.yml -p ddia_transactions down -v
```

**Plan.**

1. Deux participants et un coordinateur journalisé.
2. Une validation à deux phases réussie, pas à pas.
3. Le coordinateur tombe après la préparation : transactions en doute et verrous tenus.
4. Une transaction préparée survit au redémarrage de la base.
5. Reprise par le journal du coordinateur.
6. La décision heuristique : quand l'administrateur se trompe.
7. Le coût de 2PC en latence.

In [1]:
import json
import os
import subprocess
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
from psycopg import errors

from ddia.timelines import SpaceTime
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
BASE = "host=localhost port=10832 user=ddia password=ddia connect_timeout=5"
CONTENEUR = "ddia_transactions-postgres-1"
BANQUES = ["banque_a", "banque_b"]


def attendre_postgres(timeout=60):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with psycopg.connect(BASE + " dbname=ddia", autocommit=True) as c:
                c.execute("SELECT 1")
                return time.time() - t0
        except psycopg.OperationalError:
            time.sleep(0.5)
    raise RuntimeError("PostgreSQL ne répond pas : lancer docker compose (voir ci-dessus)")


print(f"PostgreSQL prêt ({attendre_postgres():.1f} s)")
with psycopg.connect(BASE + " dbname=ddia", autocommit=True) as c:
    print("max_prepared_transactions =", c.execute("SHOW max_prepared_transactions").fetchone()[0])
    # nettoyage d'une exécution précédente : des transactions préparées empêchent DROP DATABASE
    for gid, base_ in c.execute("SELECT gid, database FROM pg_prepared_xacts").fetchall():
        with psycopg.connect(BASE + f" dbname={base_}", autocommit=True) as c2:
            c2.execute(f"ROLLBACK PREPARED '{gid}'")
    for b in BANQUES:
        c.execute(f"DROP DATABASE IF EXISTS {b} WITH (FORCE)")
        c.execute(f"CREATE DATABASE {b}")


def connexion(banque, autocommit=False):
    return psycopg.connect(BASE + f" dbname={banque}", autocommit=autocommit)


def sql(banque, requete, params=None):
    with connexion(banque, autocommit=True) as c:
        cur = c.execute(requete, params)
        return cur.fetchall() if cur.description else None


def init_comptes():
    for b in BANQUES:
        for gid, in sql(b, "SELECT gid FROM pg_prepared_xacts WHERE database = current_database()"):
            sql(b, f"ROLLBACK PREPARED '{gid}'")
        sql(b, "DROP TABLE IF EXISTS comptes; CREATE TABLE comptes (id text PRIMARY KEY, solde int NOT NULL CHECK (solde >= 0));"
               "INSERT INTO comptes VALUES ('inès', 1000)")


def soldes():
    return {b: sql(b, "SELECT solde FROM comptes WHERE id = 'inès'")[0][0] for b in BANQUES}


def preparees():
    lignes = []
    for b in BANQUES:
        for gid, prepared in sql(b, "SELECT gid, prepared FROM pg_prepared_xacts WHERE database = current_database()"):
            lignes.append(dict(base=b, gid=gid, préparée_à=prepared.strftime("%H:%M:%S")))
    return pd.DataFrame(lignes, columns=["base", "gid", "préparée_à"])

PostgreSQL prêt (0.0 s)
max_prepared_transactions = 10


## 1. Deux participants et un coordinateur journalisé (section *Atomic Commit and Two-Phase Commit (2PC)*)

Chaque banque est une base distincte (un participant). Inès a 1 000 € dans chacune. Un
virement de 100 € de la banque A vers la banque B doit être atomique : débit **et** crédit,
ou ni l'un ni l'autre.

Les identifiants de transaction préparée (`gid`) sont uniques pour tout le serveur ; nos
deux « banques » partageant le même serveur, chaque participant reçoit l'identifiant
global suffixé de son nom (`vir-3:banque_a`).

Le coordinateur est une petite classe Python :

- il ouvre une transaction locale sur chaque participant et y fait les écritures ;
- phase 1 : `PREPARE TRANSACTION 'gid'` sur chacun (si l'un échoue, il annule tout) ;
- il écrit sa **décision** dans un journal sur disque (un fichier JSON, `fsync`) : c'est le
  point de validation ;
- phase 2 : `COMMIT PREPARED 'gid'` (ou `ROLLBACK PREPARED`) sur chacun.

L'option `panne_apres` permet de « tuer » le coordinateur à un point donné.

In [2]:
class Panne(Exception):
    pass


class Coordinateur:
    def __init__(self, journal):
        self.journal = journal

    def journaliser(self, gid, decision):
        with open(self.journal, "a") as f:
            f.write(json.dumps({"gid": gid, "décision": decision}) + "\n")
            f.flush()
            os.fsync(f.fileno())

    def decisions(self):
        if not os.path.exists(self.journal):
            return {}
        with open(self.journal) as f:
            return {d["gid"]: d["décision"] for d in map(json.loads, f)}

    def virement(self, gid, montant, panne_apres=None):
        conns = {b: connexion(b) for b in BANQUES}
        prepares = []
        try:
            try:                                              # exécution puis phase 1
                conns["banque_a"].execute("UPDATE comptes SET solde = solde - %s WHERE id = 'inès'", (montant,))
                conns["banque_b"].execute("UPDATE comptes SET solde = solde + %s WHERE id = 'inès'", (montant,))
                for b, c in conns.items():
                    c.execute(f"PREPARE TRANSACTION '{gid}:{b}'")
                    prepares.append(b)
            except psycopg.Error as e:                        # un participant refuse : abandon partout
                for b, c in conns.items():
                    c.rollback()
                for b in prepares:
                    sql(b, f"ROLLBACK PREPARED '{gid}:{b}'")
                self.journaliser(gid, "abort")
                return f"annulé ({type(e).__name__})"
            if panne_apres == "prepare":
                raise Panne
            self.journaliser(gid, "commit")                   # point de validation
            for b in BANQUES:                                 # phase 2
                if panne_apres == f"commit_{b}":
                    raise Panne
                sql(b, f"COMMIT PREPARED '{gid}:{b}'")
            return "validé"
        except Panne:
            return "PANNE DU COORDINATEUR"
        finally:
            for c in conns.values():
                c.close()


journal = os.path.join(tempfile.mkdtemp(), "coordinateur.jsonl")
coord = Coordinateur(journal)
init_comptes()
soldes()

{'banque_a': 1000, 'banque_b': 1000}